# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yuguda999/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Ranking / Scoring** — built on a **binary classification** sub-problem.

Lane 2 (Refresh / Content Opportunity Scoring, picked in `w01_research_question.ipynb`) asks "which pages should the reviewer look at first?" — that's a ranking question ("which ones first?"), not a standalone yes/no question. The way the starter pipeline (and this project) gets a ranking is by training a classifier for `is_declining_label`, taking its predicted probability as a priority signal, then blending it with a rule-based baseline score into one final score used purely to sort pages:

```
final_refresh_score = 100 * (0.70 * model_probability + 0.30 * normalized_baseline_score)
```

So classification is the modeling mechanism; ranking/scoring is the task type the actual decision needs.

In [1]:
import os
import pandas as pd

# find the repo root from wherever this kernel started (VS Code/Colab/CLI all differ)
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df["trend_direction"].value_counts())
print("\nThis feeds one ranking output: final_refresh_score = 100 * (0.70*model_probability + 0.30*normalized_baseline_score)")

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

This feeds one ranking output: final_refresh_score = 100 * (0.70*model_probability + 0.30*normalized_baseline_score)


## 2. Target or proxy

**Target:** `is_declining_label = (trend_direction == "down")`.

This is a **proxy label**, not a future-observed outcome — `trend_direction` is itself a bucket computed from `trend_pct`, a measure of the CURRENT 90-day window. It answers "is this page trending down right now," not "will this page decline next month." I'm using it because the starter CSV doesn't ship a future-window fact table; the warehouse release does. A stronger version of this project defines the target as decline over the NEXT 30 days measured from features in the PRIOR 90 days (the lane guide's decline-vs-consolidation/seasonality/noise section spells out how). Every claim built on this proxy gets flagged as "observed on the current window," not "predictive of the future."

In [2]:
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(df["is_declining_label"].value_counts())
print(f"positive rate: {df['is_declining_label'].mean() * 100:.1f}%")
df[["content_id", "client_id", "trend_direction", "is_declining_label"]].head(5)

is_declining_label
1    16262
0    13738
Name: count, dtype: int64
positive rate: 54.2%


,content_id,client_id,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,down,1
1,content_a1fb4e703a9e,client_4e07408562,down,1
2,content_9aa793d4d895,client_7f2253d7e2,down,1
3,content_331d6c4de07b,client_19581e27de,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,down,1


## 3. Success metric

**Precision@50** (already computed in this repo's `outputs/model_results.json`): of the top 50 pages the ranking puts first, how many are actually labeled declining?

Why this over accuracy or ROC-AUC alone: the decision is capacity-constrained — a reviewer opens the top N pages, not all 30,000. What matters is whether the TOP of the list is right, not whether the whole dataset is well-separated. Precision@50 measures exactly that. Average precision is my secondary check — ranking quality across all thresholds, not just the top cut.

In [3]:
import json

with open("outputs/model_results.json") as f:
    results = json.load(f)

print("baseline precision@50       :", results["baseline"]["baseline_precision_at_50"])
print("random forest precision@50  :", results["models"]["random_forest"]["precision_at_50"])
print("baseline average precision  :", round(results["baseline"]["baseline_average_precision"], 3))
print("random forest avg precision :", round(results["models"]["random_forest"]["average_precision"], 3))
print("validation split            :", results["split_strategy"])

baseline precision@50       : 0.24
random forest precision@50  : 0.68
baseline average precision  : 0.468
random forest avg precision : 0.61
validation split            : client_holdout


## 4. The unit of analysis, as a real dataframe

One row = one existing content item (a page) for one client, described by its trailing-90-day snapshot. Loaded below with the columns this lane actually uses.

In [4]:
lane_cols = [
    "content_id", "client_id", "content_type", "content_age_days",
    "impressions_90d", "avg_position", "ctr", "days_since_last_update",
    "word_count", "trend_direction", "is_declining_label",
]
lane_df = df[lane_cols]

print(f"shape: {lane_df.shape[0]:,} rows x {lane_df.shape[1]} cols")
print("one row = one (client, content item) pair")
lane_df.head(5)

shape: 30,000 rows x 11 cols
one row = one (client, content item) pair


,content_id,client_id,content_type,content_age_days,impressions_90d,avg_position,ctr,days_since_last_update,word_count,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,187,3803,10.6,0.76,20,3221.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,15320,20.3,0.05,25,2481.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,12581,36.5,0.09,20,3515.0,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,11751,6.2,0.49,22,NaN,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,19140,44.0,0.13,14,2803.0,down,1


## 5. Why ML beats a fixed rule here

The starter baseline is already a fixed, hand-tuned rule — 4 weighted components (`0.40*visibility + 0.30*freshness_risk + 0.25*position_opportunity + 0.05*depth_gap`), built from 6 single-signal reason codes. Even hand-tuned, it gets precision@50 = 0.240. The random forest, fed the same underlying signals, gets 0.680 — 2.8x. That gap is the evidence: the real risk pattern is a joint, shifting combination of trend, position, freshness, demand, and depth. A fixed threshold or fixed weight can't capture which combinations matter or how they interact (a declining page with real demand is a very different risk than a declining page nobody visits — a linear rule can't easily tell those apart without hand-built interaction terms). The model actually weighs many signals at once, not one dominant one a human could just write as an if-statement — see the spread below.

In [5]:
print(f"baseline: 4 hand-weighted components (visibility, freshness_risk, position_opportunity, depth_gap)")
print(f"model: {results['feature_count']} encoded features "
      f"({len(results['model_numeric_features'])} numeric, {len(results['model_categorical_features'])} categorical)")
print()
print("model's own top signals — spread across trend, position, age, and content depth, not one rule:")
for feat in results["best_model"]["feature_importance_top"][:6]:
    print(f"  {feat['feature']:<22} {feat['importance']:.3f}")

baseline: 4 hand-weighted components (visibility, freshness_risk, position_opportunity, depth_gap)
model: 52 encoded features (18 numeric, 8 categorical)

model's own top signals — spread across trend, position, age, and content depth, not one rule:
  days_with_impressions  0.161
  log_impressions_90d    0.128
  avg_position           0.108
  content_age_days       0.095
  word_count             0.041
  char_count             0.040


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.